In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# FlexiCodec Audio Tokenizer Example

This notebook demonstrates the FlexiCodec audio tokenizer

**Key Properties**
- Input sample rate: 16 kHz
- Output sample rate: 16 kHz 
- Number of Quanitzers for inference: 8 (upto 24)
- Semantic codebook size (FSQ): 8^5 = 32768 
- Acoustic codebook size (RVQ): (8-1) * 4096 = 28672
- Frame Rate per sec : <12.5Hz
- Token rate: <(12.5 * 8) = <100 tokens per second
- Architecture: Merge codes if vectors are similarity 
- Supports speech, audio and music

## Setup and Imports


In [1]:
import sys
import os
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our tokenizer registry
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


/users/arsaikia/benchmark-audio-tokenizer/.venv-flexicodec/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.6.0a0+df5bbc09d1.nv24.11
CUDA available: True
Using device: cuda


## Load ESB Diagnostic AMI Dataset


In [3]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean',trust_remote_code=True)

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")


Loading ESB diagnostic dataset - AMI clean split...
Dataset loaded with 770 samples
Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples


In [4]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()


Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize FlexiCodec

In [2]:
# Initialize the tokenizer
print("Loading FlexiCodec tokenizer...")
tokenizer = get_tokenizer('flexicodec', device=device, use_speaker_encoder=False)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")
print(f"  Token rate: ~{tokenizer.sample_rate / tokenizer.downsample_rate:.1f} Hz")


Loading FlexiCodec tokenizer...


Fetching 29 files: 100%|██████████| 29/29 [00:00<00:00, 42.11it/s]
/users/arsaikia/benchmark-audio-tokenizer/.venv-flexicodec/lib/python3.12/site-packages/audiotools/core/audio_signal.py:32: SyntaxWarning: invalid escape sequence '\_'
  """
/users/arsaikia/benchmark-audio-tokenizer/.venv-flexicodec/lib/python3.12/site-packages/audiotools/core/audio_signal.py:1012: SyntaxWarning: invalid escape sequence '\_'
  """Wrapper around scipy.signal.get_window so one can also get the
/users/arsaikia/benchmark-audio-tokenizer/.venv-flexicodec/lib/python3.12/site-packages/audiotools/core/audio_signal.py:1092: SyntaxWarning: invalid escape sequence '\_'
  """Compute how the STFT should be padded, based on match\_stride.
/users/arsaikia/benchmark-audio-tokenizer/.venv-flexicodec/lib/python3.12/site-packages/audiotools/core/audio_signal.py:1131: SyntaxWarning: invalid escape sequence '\_'
  """Computes the short-time Fourier transform of the audio data,
/users/arsaikia/benchmark-audio-tokenizer/.venv

Notice: ffmpeg is not installed. torchaudio is used to load audio
If you want to use ffmpeg backend to load audio, please install it by:
	sudo apt install ffmpeg # ubuntu
	# brew install ffmpeg # mac


funasr version: 1.3.1.
Loading remote code successfully: /users/arsaikia/benchmark-audio-tokenizer/examples/../src/audio_tokenizers/implementations/../../repos/FlexiCodec/flexicodec/customized_sensevoice/model.py
FlexiCodec - Total Parameters: 448.94M
FlexiCodec - Trainable Parameters: 214.94M
Tokenizer loaded: FlexiCodecWrapper(checkpoint='None', device='cuda', sample_rate=16000)
  Input sample rate: 16000 Hz
  Output sample rate: 16000 Hz
  Codebook size: 32,768
  Downsample rate: 1280.0x
  Token rate: ~12.5 Hz


In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "128"

# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor and ensure correct shape (1, T) for WavTokenizer
    audio_tensor = torch.from_numpy(audio_array).float()
    if audio_tensor.dim() == 1:
        audio_tensor = audio_tensor.unsqueeze(0)  # (T,) -> (1, T)

    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)
    
    # Show token information
    B, C, T = tokens.shape  # Batch, Codebooks, Time

    total_semantic_tokens = B * tokens.sep * T
    total_acoustic_tokens = B * (C - tokens.sep) * T

    print(f"Token shape: {tokens.shape}")
    print(f"Total Number of tokens: {tokens.numel()}")
    print(f"Tokens per second: {tokens.numel() / duration:.1f}")

    # Corrected Semantic/Acoustic stats
    print(f"Semantic Tokens per second: {total_semantic_tokens / duration:.1f}")
    print(f"Acoustic Tokens per second: {total_acoustic_tokens / duration:.1f}")
    print(f"Total Semantic Tokens: {total_semantic_tokens}")
    print(f"Total Acoustic Tokens: {total_acoustic_tokens}")

    # # Show first 20 discrete token values
    token_values = tokens.squeeze().cpu().numpy()
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")

    print("\nDecoding tokens back to audio...")
    reconstructed, decode_info = tokenizer.decode(tokens)
    
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Output sample rate: {decode_info['output_sample_rate']} Hz")
    
    # Handle shape: (1, 1, T) or (1, T) or (T,)
    rec = reconstructed
    if rec.dim() == 3:
        rec = rec[0, 0] if rec.shape[1] == 1 else rec[0]
    elif rec.dim() == 2:
        rec = rec[0]
    rec_np = rec.detach().cpu().numpy() if torch.is_tensor(rec) else rec
    
    # Store results
    results.append({
        'original': audio_array,
        'original_sr': sr,
        'reconstructed': rec_np,
        'reconstructed_sr': decode_info['output_sample_rate'],
        'tokens': token_values,
        'transcript': transcript,
        'duration': duration
    })
    
print(f"\n{'='*60}")
print("Processing complete!")



Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...
Token shape: torch.Size([1, 8, 118])
Total Number of tokens: 944
Tokens per second: 80.2
Semantic Tokens per second: 10.0
Acoustic Tokens per second: 70.2
Total Semantic Tokens: 118
Total Acoustic Tokens: 826

First 20 discrete token IDs:
[[22847 31407 14961  3182 30986  5077  9891 28834  4649 22267 20890  5372
  11628 16292  1618  5375 21474 14507 22307  8774 21553  4714 10727  6823
   9966 13872  8253 28510 10523 30390 16051 10550  1343 22013 20396  1853
  19948 13029  1789  6824  9265 19763 25378  9393 24381 32584 18611 28325
  21110 10236 23353 15011 25442 26742 18101 12599 20345 12511 23352 28705
  10420 23285 31298  5427 15136 12941  8357 29932  8289 29090  6160  3830
  19121  2292 25090  5289  8163  3875  6448 10907 12951 18356 26869 28488
   958

In [29]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    print(f"\nOriginal Audio ({result['original_sr']} Hz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    print(f"\nReconstructed Audio ({result['reconstructed_sr']} Hz):")
    display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # 12-bit tokens (log2(4096) = 12)
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")



Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 8 (0.7 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 23540.0x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 8 (0.7 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 23400.0x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 8 (0.7 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 23380.0x


## Summary

Flexicodec successfully:
- Encodes audio to discrete tokens at <12.5Hz frame rate dynamically
- Uses FSQ for quantization of semantic tokens. (Size: 8^5 = 32768)
- Uses RVQ for quantization of acoustic tokens. (Size: (8-1) * 4096 = 28672)
- Supports speech, audio, and music

The discrete tokens can be used for:
- Audio compression and storage
- Input to language models for audio generation
- Audio editing and manipulation tasks

**Key Advantages:**
- **<12.5 frames/second**: Very low frame rate
- **Universal support**: Works well with speech, audio and music